# RoBERTa Sentiment Analysis
Setup environment and initialize a RoBERTa-based sentiment pipeline.

In [11]:
# Install training dependencies
#import sys
#!{sys.executable} -m pip install -U datasets scikit-learn

In [12]:
# Load dataset from CSV and prepare splits (auto-detect columns)
from datasets import load_dataset, ClassLabel, Value

csv_path = r"d:\NLP_Project\twitter_sentiment_data.csv"
dataset = load_dataset("csv", data_files={"train": csv_path}, split="train")

# Auto-detect text and label columns using preview and features
preview = dataset.select(range(min(1000, len(dataset))))
cols = preview.column_names
feats = getattr(preview, "features", {})

preferred_text = ["text", "tweet", "review", "content", "message", "body"]
preferred_label = ["label", "labels", "sentiment", "target", "polarity", "class"]

# Choose text column
text_column = None
for c in preferred_text:
    if c in cols:
        text_column = c
        break
if text_column is None:
    for c in cols:
        f = feats.get(c)
        if isinstance(f, Value) and getattr(f, "dtype", None) in ("string", "large_string"):
            text_column = c
            break

# Choose label column
label_column = None
for c in preferred_label:
    if c in cols:
        label_column = c
        break
if label_column is None:
    # Prefer ClassLabel if present
    for c in cols:
        f = feats.get(c)
        if isinstance(f, ClassLabel):
            label_column = c
            break
if label_column is None:
    # Fallback: low-cardinality integer column
    for c in cols:
        f = feats.get(c)
        if isinstance(f, Value) and getattr(f, "dtype", None) in ("int8", "int16", "int32", "int64"):
            try:
                if len(set(preview[c])) <= 10:
                    label_column = c
                    break
            except Exception:
                pass

if text_column is None or label_column is None:
    raise ValueError(f"Could not detect text/label columns. Found columns: {cols}. Please set text_column/label_column manually.")

# Create validation split (10%)
dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_ds = dataset["train"]
val_ds = dataset["test"]

print({"text_column": text_column, "label_column": label_column})
# Inspect a sample
train_ds[0]

{'text_column': 'message', 'label_column': 'sentiment'}


{'sentiment': 1,
 'message': 'The sea floor is sinking under the weight of climate change https://t.co/R9Uhnjfg7G',
 'tweetid': 954625951685578752}

In [13]:
# Map labels to a fixed 4-class scheme
def to_four_class(v):
    v = int(v)
    if v == -1: return 0  # anti
    if v == 0:  return 1  # neutral
    if v == 1:  return 2  # pro
    if v == 2:  return 3  # news
    return 1              # fallback: neutral

# Create 'labels' column with 4-class ids
train_ds = train_ds.map(lambda b: {"labels": [to_four_class(x) for x in b[label_column]]}, batched=True)
val_ds   = val_ds.map(lambda b: {"labels": [to_four_class(x) for x in b[label_column]]},   batched=True)

# Remove original label column to avoid confusion
if label_column != "labels":
    train_ds = train_ds.remove_columns([label_column])
    val_ds = val_ds.remove_columns([label_column])

# Set explicit mappings
label2id = {"anti": 0, "neutral": 1, "pro": 2, "news": 3}
id2label = {0: "anti", 1: "neutral", 2: "pro", 3: "news"}
num_labels = 4

print({"num_labels": num_labels, "id2label": id2label})

{'num_labels': 4, 'id2label': {0: 'anti', 1: 'neutral', 2: 'pro', 3: 'news'}}


In [14]:
# Tokenize dataset using a lighter RoBERTa model and shorter sequences
from transformers import AutoTokenizer, DataCollatorWithPadding
from tqdm.auto import tqdm  # works in VS Code or classic Jupyter

# Use a lighter model and shorter sequences
MODEL_NAME = "distilroberta-base"  # lighter than roberta-base
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess(batch):
    return tokenizer(batch[text_column], truncation=True, max_length=128)  # shorter seq

# Compute which columns to remove (drop text, keep labels)
cols_to_remove = [c for c in train_ds.column_names if c != "labels"]

train_tok = train_ds.map(preprocess, batched=True, remove_columns=cols_to_remove)
val_tok   = val_ds.map(preprocess,   batched=True, remove_columns=[c for c in val_ds.column_names if c != "labels"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

d:\NLP_Project\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\agari\.cache\huggingface\hub\models--distilroberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Map:   0%|          | 0/39548 [00:00<?, ? examples/s]

Map:   0%|          | 0/4395 [00:00<?, ? examples/s]

In [ ]:
# Speed hints (place in a small cell before training)
import torch
torch.backends.cudnn.benchmark = True
print("CUDA:", torch.cuda.is_available(), "Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
# Configure model, metrics, and Trainer
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

# label2id/id2label were defined earlier; ensure they exist
assert label2id is not None and id2label is not None and num_labels is not None

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    label2id={v: k for k, v in label2id.items()},
    id2label=id2label,
    ignore_mismatched_sizes=True
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }

# Speed up and reduce memory
args = TrainingArguments(
    output_dir="runs/roberta-sentiment",
    learning_rate=2e-5,
    per_device_train_batch_size=2,     # tiny batch fits 2GB VRAM
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,     # effective batch size ~= 8
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=100,
    disable_tqdm=False,

    # Memory/perf helpers
    fp16=True,                         # mixed precision on CUDA
    gradient_checkpointing=True,       # save VRAM
    dataloader_pin_memory=True,
    dataloader_num_workers=0,          # avoid CPU contention on i5 mobile
    remove_unused_columns=True,

    # Keep training simple (eval later)
    do_eval=False,                     # skip eval during train
    save_steps=0,                      # skip step-based checkpoints
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=val_tok,              # still available for a separate evaluate() call
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()
metrics = trainer.evaluate()  # run once after training
print(metrics)

trainer.save_model("models/roberta-sentiment-finetuned")
tokenizer.save_pretrained("models/roberta-sentiment-finetuned")

CUDA: False Device: CPU


model.safetensors:   0%|          | 0.00/331M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at distilroberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\agari\AppData\Local\Temp\ipykernel_15388\3016532897.py:53: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
C:\Users\agari\AppData\Local\Temp\ipykernel_15388\3016532897.py:53: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
100,1.137700


In [ ]:
# Initialize RoBERTa sentiment pipeline using CardiffNLP model or fine-tuned model
import os
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TextClassificationPipeline
import torch

MODEL_NAME = "models/roberta-sentiment-finetuned" if os.path.isdir("models/roberta-sentiment-finetuned") else "cardiffnlp/twitter-roberta-base-sentiment-latest"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

# Build a TextClassificationPipeline
sentiment_pipeline = TextClassificationPipeline(
    model=model,
    tokenizer=tokenizer,
    framework="pt",
    device=0 if torch.cuda.is_available() else -1
)

labels = model.config.id2label
labels

In [ ]:
# Optional: install required packages (uncomment if needed)
## Note: Running installs from the notebook may require internet access.
## On Windows cmd, you can also install via terminal:
## pip install --upgrade pip
## pip install transformers torch sentencepiece

# If you prefer inline install, uncomment the following:
# import sys
# !{sys.executable} -m pip install -U pip
# !{sys.executable} -m pip install transformers torch sentencepiece

In [ ]:
# Fix notebook progress bars and optional HF Xet support
# import sys
# !{sys.executable} -m pip install --upgrade ipywidgets jupyter jupyterlab notebook
# Optional: speed up Hugging Face downloads
# !{sys.executable} -m pip install "huggingface_hub[hf_xet]"
# Optional: silence symlink warning without enabling Developer Mode
# import os
# os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

In [ ]:
# Quick test
texts = [
    "I love this product!",
    "This is the worst experience ever.",
    "It's okay, nothing special.",
    "I hate you so much!",
    "I adore you so much!"
]

results = sentiment_pipeline(texts, top_k=1)
for text, res in zip(texts, results):
    print(f"{text} -> {res[0]['label']} ({res[0]['score']:.3f})")